# Transformer Triton kernel — Colab runner

Use a GPU runtime. This notebook invokes the same tests, manifest runner, and profiler as local validation. Colab results describe the assigned Colab GPU, not the curated RTX 5070 Ti run.

## 1. Clone the complete repository

Individual-file upload is insufficient because the kernel, dispatcher, manifests, and tools are separate modules.

In [ ]:
import os
if not os.path.isdir('/content/tiktok-techjam-2026'):
    !git clone https://github.com/lukeai-tan/tiktok-techjam-2026.git /content/tiktok-techjam-2026
%cd /content/tiktok-techjam-2026

## 2. Verify CUDA, PyTorch, and Triton

In [ ]:
import torch, triton
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('triton', triton.__version__)
print('gpu', torch.cuda.get_device_name(), 'capability', torch.cuda.get_device_capability())

## 3. Run the CPU/GPU contract suite

A compiler and Python headers may be required when Colab first builds Triton's driver shim.

In [ ]:
!python -m pip install -q pytest==9.1.1
!python -m pytest tests -q

## 4. Fast manifest smoke run

In [ ]:
!python benchmarks/run_matrix.py --device cuda --case long-causal-padding --dtype float32 --attention-backend auto --quick --accuracy-trials 3 --out results/colab-smoke.json

## 5. Full provisional matrix

This fails unless every requested case is PASS. It never converts compilation errors or OOM-only runs into success.

In [ ]:
!python benchmarks/run_matrix.py --device cuda --attention-backend auto --accuracy-trials 5 --out results/colab-matrix.json

## 6. Profiler proof

In [ ]:
!python benchmarks/profile_cases.py --case long-causal-padding --dtype float32 --attention-backend auto --steps 5 --out results/colab-profile.json

## 7. Download evidence

In [ ]:
from google.colab import files
files.download('results/colab-matrix.json')
files.download('results/colab-profile.json')